# posthoc_geometry: training-free geometric edits of the finished main-line student

Curiosity experiment, no training. Start: the 1k-base student distilled from BGE-M3 + instructed 8K-embed (10,800 steps, lr 1e-4; TR-MTEB 67.71). The statistics of its own embeddings on 20,000 texts of the training mix (not benchmark texts) give a linear map that is **folded into the model as one extra identity-activation Dense layer** (`turkish_distill.posthoc`), optionally followed by Normalize. Four variants are written to Drive `models/posthoc_<name>`; evaluate them with the usual eval notebooks (plain protocol) and compare with 67.71.

- `center`: subtract the mean embedding (no Normalize, vectors keep their scale)
- `center_norm`: the same, then Normalize
- `abtt5_norm`: centre, project out the top 5 principal directions ("all but the top", Mu and Viswanath 2018), Normalize
- `zca_norm`: centre, ZCA-whiten with eigenvalue shrinkage (eps = 0.1 of the mean eigenvalue), Normalize

What to expect, written down before the results: centring changes only cosine-based tasks (retrieval, STS, pair classification, bitext); k-means and logistic regression are translation-invariant, so clustering and classification should hardly move unless Normalize or whitening acts. Whitening can hurt retrieval. Earlier, Normalize alone changed the student by -0.3..+0.4 per category. A gain above 0.3 on Mean(task) would be news; interpret smaller differences as noise (seed noise ~0.1 for training, evaluation itself is deterministic).

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'      # local disk, never Drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib, importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell first.')
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')       # an editable install's .pth is not read by a running session
importlib.invalidate_caches()
os.environ.setdefault('HF_HOME', '/content/hf_cache')
import turkish_distill.posthoc                  # fails here if the repo on GitHub does not have the module yet

In [ ]:
# --- statistics of the model's own embeddings on training-mix texts (GPU, a few minutes) ---
import json, random
import numpy as np
from sentence_transformers import SentenceTransformer
from turkish_distill.posthoc import append_linear_head, apply_map, fit_stats, linear_map, mean_pairwise_cosine

MODEL = f'{DRIVE_ROOT}/runs/scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b/final'   # main line (TR-MTEB 67.71)
MIX = f'{DRIVE_ROOT}/data/mix/mix_v2_200k.jsonl'
OUT_MODELS = f'{DRIVE_ROOT}/models'
N_FIT, N_HELD, K, EPS = 20000, 2000, 5, 0.1
assert os.path.isdir(MODEL), MODEL

texts = [json.loads(line)['text'] for line in open(MIX, encoding='utf-8')]
random.Random(0).shuffle(texts)
fit_texts, held_texts = texts[:N_FIT], texts[N_FIT:N_FIT + N_HELD]      # disjoint: statistics from one, checks on the other
st = SentenceTransformer(MODEL, device='cuda')
st.max_seq_length = 512                                                  # the TR-MTEB protocol
encode = lambda t: st.encode(t, batch_size=128, convert_to_numpy=True, show_progress_bar=True)
E_fit, E_held = encode(fit_texts), encode(held_texts)
stats = fit_stats(E_fit)
share = stats['eigvals'] / stats['eigvals'].sum()
print(f'{len(fit_texts)} fit texts, {len(held_texts)} held-out; mean vector norm {np.linalg.norm(E_held, axis=1).mean():.2f}')
print('variance share of the top-1 / top-5 / top-20 directions: '
      f'{share[:1].sum():.3f} / {share[:5].sum():.3f} / {share[:20].sum():.3f}')

In [ ]:
# --- anisotropy before and after each variant (held-out texts) ---
VARIANTS = {
    'center':      dict(kind='center', normalize=False),
    'center_norm': dict(kind='center', normalize=True),
    'abtt5_norm':  dict(kind='abtt', normalize=True),
    'zca_norm':    dict(kind='zca', normalize=True),
}
print(f'{"original":<14} mean pairwise cosine {mean_pairwise_cosine(E_held):.3f}')
for name, v in VARIANTS.items():
    w, b = linear_map(v['kind'], stats, k=K, eps=EPS)
    y = apply_map(E_held, w, b)
    print(f'{name:<14} mean pairwise cosine {mean_pairwise_cosine(y):.3f}')

In [ ]:
# --- write the four models to Drive and check each one against the numpy map after a save/reload round trip ---
for name, v in VARIANTS.items():
    out = f'{OUT_MODELS}/posthoc_{name}'
    if os.path.isdir(out):
        print('exists, skipped:', out)
        continue
    w, b = linear_map(v['kind'], stats, k=K, eps=EPS)
    append_linear_head(st, w, b, v['normalize']).save(out)
    reloaded = SentenceTransformer(out, device='cuda')
    reloaded.max_seq_length = 512
    got = reloaded.encode(held_texts[:64], batch_size=64, convert_to_numpy=True)
    want = apply_map(E_held[:64], w, b)
    if v['normalize']:
        want = want / np.linalg.norm(want, axis=1, keepdims=True)
    cos = (got * want).sum(1) / (np.linalg.norm(got, axis=1) * np.linalg.norm(want, axis=1))
    print(f'{name:<14} saved; reloaded model vs numpy map: min cosine {cos.min():.5f}')
    assert cos.min() > 0.999, name
    del reloaded

print()
for name in VARIANTS:
    print(f"    'posthoc_{name}': dict(model=f'{{DRIVE_ROOT}}/models/posthoc_{name}'),")
print('\nPaste these preset lines into the eval notebook (plain protocol, no extra normalize flag: the Normalize layer is inside the model where wanted).')

In [ ]:
from google.colab import runtime
runtime.unassign()